# lsdsk Quickstart

Install the package (from PyPI or GitHub), then try the CLI and API examples below.

## 1) Install latest from GitHub (default)

In [ ]:
import sys

# Install uv into this kernel's environment first: a bare `!uv` needs uv on
# PATH, which a reader running this notebook from a plain Python will not have.
!{sys.executable} -m pip install --quiet --upgrade uv

# install lsdsk from GitHub (default)
!{sys.executable} -m uv pip install --python {sys.executable} --upgrade git+https://github.com/bitranox/lsdsk.git

## Or: Install from PyPI (alternative)
Uncomment and run the cell below if you prefer the PyPI release.

In [ ]:
# !{sys.executable} -m pip install --upgrade lsdsk


## 2) Version and help


In [ ]:
!lsdsk --version
assert _exit_code == 0

## 3) Scan this machine

The default view: a problem summary above the topology tree.
Exit code 1 simply means something actionable was found.

A bare `lsdsk` at a terminal opens the interactive view instead; `--report`
asks for this page, which is the form a notebook can render.


In [ ]:
!lsdsk --report
assert _exit_code in (0, 1)

## 4) Replay a capture from another machine

A snapshot holds the raw reading, so replaying it runs the same decoding and
diagnosis a live scan does. This one holds 19 disks across two SAS host bus
adapters, AHCI and NVMe.


In [ ]:
# Resolve the fixture from either working directory: a reader runs this
# from notebooks/, CI runs it from the repository root.
from pathlib import Path

SNAPSHOT = next(
    str(candidate)
    for candidate in (
        Path("tests/fixtures/hw/linux-sas-hba.json"),
        Path("../tests/fixtures/hw/linux-sas-hba.json"),
    )
    if candidate.exists()
)
!lsdsk --report --replay {SNAPSHOT}
assert _exit_code in (0, 1)

## 5) Every finding, with its reasoning and its remedy


In [ ]:
!lsdsk findings --replay {SNAPSHOT}
assert _exit_code in (0, 1)

## 6) Health: wear, temperature and error counters


In [ ]:
!lsdsk health --replay {SNAPSHOT}
assert _exit_code in (0, 1)

## 7) Controllers and their PCIe placement


In [ ]:
!lsdsk controllers --replay {SNAPSHOT}
assert _exit_code in (0, 1)

## 8) Machine-readable output


In [ ]:
# The JSON envelope is what another program consumes. Read it here with
# the library rather than a subprocess, which is the same data and keeps
# the notebook free of shell plumbing.
import json

from lsdsk.adapters.cli.commands.scan import build_envelope
from lsdsk.adapters.hw import snapshot
from lsdsk.domain.diagnostics import diagnose
from lsdsk.domain.enums import CliCommand

inventory = snapshot.load(Path(SNAPSHOT))
# `command` is required and has no default: a default here once let every
# command label its output `scan`. The envelope is a model, so read it with
# model_dump() rather than subscripting it.
envelope = build_envelope(inventory, diagnose(inventory), CliCommand.FINDINGS)
payload = envelope.model_dump()
print(json.dumps(payload["data"]["findings"][0], indent=2, default=str))

## 9) The library API

The same inventory and rules the CLI uses.


In [ ]:
from lsdsk.adapters.hw import snapshot
from lsdsk.domain.diagnostics import diagnose

inventory = snapshot.load(__import__("pathlib").Path(SNAPSHOT))
findings = diagnose(inventory)

print(inventory.hostname, len(inventory.controllers), "controllers")
for finding in findings[:3]:
    print(f"[{finding.severity}] {finding.subject}: {finding.title}")